# Taxonomy Leaf Assignment - Mistral Batch

Notebook minimal pour assigner les `paraphrase_task` du set de clustering (`mistral_extraction_full`, single-record, non-multitask) à la taxonomie manuelle avec Mistral Batch.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve().parent))

In [ ]:
from src.taxonomy.providers import mistral_batch_results, mistral_batch_submit
from src.taxonomy.schemas import TASKS, VALID_TAXONOMY

## 1. Config

In [ ]:
TASK = "taxonomy_leaf_assignment"
MODEL = "mistral-large-latest"
RUN_NAME = "clustering_set"

DATA_DIR = Path("../data")
TAXONOMY_DIR = DATA_DIR / "taxonomy"
PER_LLM_DIR = TAXONOMY_DIR / "per_llm"
OUT_DIR = TAXONOMY_DIR / "leaf_assignments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / f"{RUN_NAME}_mistral_leaf_assignments.parquet"

cfg = TASKS[TASK]
print(cfg.tool_name, cfg.max_tokens)
print(
    "taxonomy:",
    len(VALID_TAXONOMY),
    "mothers,",
    sum(len(v) for v in VALID_TAXONOMY.values()),
    "leaves",
)

## 2. Load Clustering Set

In [ ]:
df = pd.read_parquet(PER_LLM_DIR / "mistral_extraction_full.parquet").merge(
    pd.read_parquet(PER_LLM_DIR / "google.parquet")[
        ["bibkey", "google_task_type", "title", "abstract"]
    ],
    on="bibkey",
)

print(f"After load+merge: {len(df):,}")

required = {"bibkey", "title", "abstract"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["is_multi_record"] = df.groupby("bibkey")["bibkey"].transform("size") > 1
df = df[~df.is_multi_record]
print(f"After multi-record filter: {len(df):,}")

df = df[(~df.is_multitask) & (~df.subtasks_not_inferable)]
print(f"After multitask filter: {len(df):,}")

df = df.dropna(subset=["bibkey", "title", "abstract"]).sort_values("bibkey")
df = df.reset_index(drop=True)

df[["bibkey", "title", "abstract", "paraphrase_task"]].head(3)

In [ ]:
# Temporary smoke test subset
# TEST_N = 20
# df = df.sample(TEST_N, random_state=42).sort_values("bibkey").reset_index(drop=True)
# df = df.head(TEST_N).reset_index(drop=True)
print(f"Smoke test subset: {len(df):,} papers")

## 3. Submit Batch

Run once. Keep the printed `job_id`.

In [ ]:
items = [
    (
        str(row.bibkey),
        {"paper_text": row.paraphrase_task},
    )
    for row in df.itertuples(index=False)
]

job_id = mistral_batch_submit(items, model=MODEL, task=TASK)
job_id

## 4. Fetch Results

À lancer quand le batch Mistral est terminé. Les assignations invalides `mother`/`leaf` sont filtrées pendant le fetch.

In [ ]:
JOB_ID = "47d3de90-340c-499b-8dc9-78b83ff7c695"

assigned = mistral_batch_results(JOB_ID, df, task=TASK)
assigned.to_parquet(OUT_PATH, index=False)
OUT_PATH

## 5. Quick Inspection

In [ ]:
mother_col = "mistral_mother"
leaf_col = "mistral_leaf"

print("rows:", len(assigned))
missing_assignments = assigned[mother_col].isna().sum()
print("missing/invalid assignments:", missing_assignments)

display(assigned[mother_col].value_counts(dropna=False).head(30))
display(assigned[["bibkey", "title", mother_col, leaf_col]].head(20))